In [ ]:
import torch.nn as nn
import torch
import torch.optim as optim

In [ ]:
# temperature data in celcius
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]

# temperature in unknown units
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

# each row is meant to represent a new data "point", but
# our data is in a single row, hence its dimension is [11]
# but we want it to be 11x1, so we insert a singleton dimension
t_c.unsqueeze_(1)
t_u.unsqueeze_(1)

n_samples = t_u.shape[0]
# 20% goes to validation
n_val = int(0.2 * n_samples)

# generate 0 through n_samples-1, randomly permutated
shuffled_indices = torch.randperm(n_samples)

# train on all but the last 20%
train_indices = shuffled_indices[:-n_val]

# validate on the final 20%
val_indices = shuffled_indices[-n_val:]

# make the train and test data sets
train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]

val_t_u = t_u[val_indices]
val_t_c = t_c[val_indices]

# normalize
train_t_un  = train_t_u * 0.1
val_t_un = val_t_u * 0.1

In [ ]:
linear_module = nn.Linear(1,1) # default bias is True
linear_module(val_t_un)

In [ ]:
linear_module.weight

In [ ]:
linear_module.bias

In [ ]:
x = torch.ones(1)
linear_module(x)

In [ ]:
x = torch.ones(10,1)
linear_module(x)

In [ ]:
linear_model = nn.Linear(1,1)
optimizer = optim.SGD(
    linear_model.parameters(),
    lr = 1e-2
) 

In [ ]:
def training_loop(n_epochs,optimizer,model,loss_fn,
                t_u_train,t_u_val,t_c_train,t_c_val):
    for epoch in range(1,n_epochs+1):
        # evaluate the model on the training data
        t_p_train = model(t_u_train)

        # evaluate the training loss
        loss_train = loss_fn(t_p_train, t_c_train)

        # evaluate the model on the validation data
        t_p_val = model(t_u_val)

        # evaluate the validation loss
        loss_val = loss_fn(t_p_val, t_c_val)

        # clear the gradients
        optimizer.zero_grad() #parameters as "stored" in the optimizer class

        # calcuate the gradients on the training data
        loss_train.backward() # gradients are calculated on the loss function

        # update the parameters given training data gradients
        optimizer.step() # parameters are "stored" and hence updated in the optimizer class

        if epoch ==1 or epoch % 1000 == 0:
            print(f"Epoch {epoch}, Training Loss {loss_train.item():.4f},"
                  f" Validation Loss {loss_val:.4f}")


In [ ]:
def loss_fun(t_c, t_u):
    sqr_error = (t_c - t_u)**2
    return sqr_error.mean()

In [ ]:
training_loop(
    n_epochs = 3000,
    optimizer = optimizer,
    model = linear_model, 
    loss_fn = loss_fun,
    t_u_train = train_t_un,
    t_u_val = val_t_un,
    t_c_train = train_t_c,
    t_c_val = val_t_c
)

In [ ]:
print(linear_model.weight)
print(linear_model.bias)